# 1　数据处理

按论文顺序整理第 II-A 数据来源、II-B 实体名称标准化与关系整合、II-C 证候桥接与候选关联生成、
II-D 前六项结构特征、II-E 专家参考标注。产出表 II、候选关系概况、三层网络、全库候选对的
结构特征表，以及 243 对专家参考标注。

第 7 项语义特征由大语言模型判定，单独放在 2 号文件里。

In [ ]:
import os
from pathlib import Path

# 找到工程根目录（含「数据」文件夹的那一层）
ROOT = Path(os.path.abspath(".")).resolve()
while ROOT != ROOT.parent and not (ROOT / "数据").is_dir():
    ROOT = ROOT.parent
DATA = ROOT / "数据"
RES = ROOT / "结果"
RES.mkdir(exist_ok=True)

from collections import Counter, defaultdict

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

ANNO_DIR = DATA / "anno文件"        # .anno 标注语料
STD_DIR = DATA / "标准化文件"        # 症状/病机/证候 标准化映射表 + 国标证候词表
GOLD_DIR = DATA / "专家标注"         # 专家评分记录与参考标注

REL_SYND_TO_SYMP = "证候典型症状"     # 证候 → 症状
REL_SYND_TO_PATH = "机理为"           # 证候 → 病机

LABEL_MAP = {"极强关联": 5, "强关联": 4, "中等关联": 3, "弱关联": 2, "极弱关联": 1}
KEY_COLS = ["症状", "病机"]
FEATURE_COLS = ["shared_syndrome_count_norm", "NPMI_norm", "Jaccard_norm",
                "AA_norm", "RA_norm", "PPR_norm"]
PPR_ALPHA, PPR_MAX_ITER = 0.85, 100   # 个性化 PageRank 的阻尼系数与迭代上限


## 1.1　解析 .anno 标注语料（II-A）

每行一条记录。Entity 行给出实体编号、类别、字符位置和名称，Relation 行给出关系编号、
关系名和两端实体编号。实体编号只在同一分卷内唯一，所以按「来源文件 + 实体标识」定位实体。

In [ ]:
def parse_anno_files(data_dir):
    """批量解析 .anno，返回实体表和关系表。"""
    anno_files = [f for f in os.listdir(data_dir) if f.endswith(".anno")]
    if not anno_files:
        raise FileNotFoundError(f"{data_dir} 下没找到 .anno 文件")
    print(f"解析 .anno 文件，共 {len(anno_files)} 个")

    entities, relations = [], []
    for fname in sorted(anno_files):
        with open(os.path.join(data_dir, fname), encoding="utf-8") as f:
            for line in f:
                parts = [p.strip() for p in line.split() if p.strip()]
                if not parts:
                    continue
                if parts[0] == "Entity" and len(parts) >= 6:
                    raw_cat = parts[2]
                    clean_cat = raw_cat.replace("中成药-", "") \
                        if raw_cat.startswith("中成药-") else raw_cat
                    entities.append([parts[1], raw_cat, clean_cat,
                                     int(parts[3]), int(parts[4]), parts[5], fname])
                elif parts[0] == "Relation" and len(parts) >= 5:
                    relations.append([parts[1], parts[2], parts[3], parts[4], fname])

    df_entities = pd.DataFrame(entities, columns=[
        "实体标识", "原始实体所属类别", "实体所属类别",
        "实体的字符开始顺序", "实体的字符结束顺序", "实体具体名称", "数据来源文件"])
    df_relations = pd.DataFrame(relations, columns=[
        "关系标识", "关联关系", "实体1", "实体2", "数据来源文件"])
    print(f"实体 {len(df_entities)} 个，关系 {len(df_relations)} 条")
    return df_entities, df_relations


df_entities, df_relations = parse_anno_files(ANNO_DIR)
print(df_relations["关联关系"].value_counts().to_dict())

## 1.2　套用实体名称标准化映射（II-B）

`数据/标准化文件/` 下三张「原始名 → 标准化名」映射表。证候名按国家标准 GB/T 16751.2—2021
的证候词表归一；症状和病机没有完全对应的国家标准，同义名归并由大语言模型辅助完成，
结果已冻结在这三张表里，这里直接读。

In [ ]:
def load_std_maps(std_folder):
    """读症状、病机、证候三张标准化映射表。"""
    std_map = {}
    for cat in ["症状", "病机", "证候"]:
        path = os.path.join(std_folder, f"{cat}.xlsx")
        if not os.path.exists(path):
            raise FileNotFoundError(f"没找到标准化文件: {path}")
        df = pd.read_excel(path, dtype=str).fillna("")
        col_new = "标准化名" if "标准化名" in df.columns else "标准化后名称"
        std_map[cat] = dict(zip(df["原始名"].astype(str).str.strip(),
                                df[col_new].astype(str).str.strip()))
        print(f"载入 {cat} 标准化映射 {len(std_map[cat])} 条")
    return std_map


def apply_standardization(df_entities, std_map):
    """按实体类别把名称换成标准名。"""
    def _map(row):
        m = std_map.get(str(row["实体所属类别"]).strip(), {})
        norm = m.get(str(row["实体具体名称"]).strip(), "")
        return norm if norm and norm.lower() != "nan" else "未识别"
    df_entities["标准化名称"] = df_entities.apply(_map, axis=1)
    return df_entities


std_map = load_std_maps(STD_DIR)
df_entities = apply_standardization(df_entities, std_map)
print("未能识别的实体:", int((df_entities["标准化名称"] == "未识别").sum()))

## 1.3　合法关系与去重计数（II-B、表 II）

只保留「证候典型症状」和「机理为」两类关系，并统计按原始实体名去重后的条数。

In [ ]:
def count_by_raw_name(df_entities, df_relations):
    """统计两类合法关系的条数，以及按原始实体名去重后的条数。"""
    eid2name = {}
    for _, r in df_entities.iterrows():
        eid2name[(str(r["数据来源文件"]).strip(), str(r["实体标识"]).strip())] = \
            str(r["实体具体名称"]).strip()

    rows = []
    for _, r in df_relations.iterrows():
        rel = str(r["关联关系"]).strip()
        if rel not in (REL_SYND_TO_SYMP, REL_SYND_TO_PATH):
            continue
        f = str(r["数据来源文件"]).strip()
        n1 = eid2name.get((f, str(r["实体1"]).strip()))
        n2 = eid2name.get((f, str(r["实体2"]).strip()))
        if n1 and n2:
            rows.append((rel, n1, n2))
    return pd.DataFrame(rows, columns=["关联关系", "实体1名称", "实体2名称"])


rel_valid = count_by_raw_name(df_entities, df_relations)
n_raw = len(rel_valid)
n_dedup = len(rel_valid.drop_duplicates())
print("合法关系 %d 条；按原始名称去重后 %d 条" % (n_raw, n_dedup))

## 1.4　构建症状—证候—病机三层网络（II-B）

把标准化后的关系整理成「证候 → 症状」和「证候 → 病机」两个邻接表，指向同一实体对的关系
用集合去重，得到标准化关系集。网络节点带类型标签，按无向图处理。

In [ ]:
def build_syndrome_maps(df_entities, df_relations):
    """构建 证候→症状 与 证候→病机 两个邻接表（标准名口径）。"""
    file_eid_map = defaultdict(dict)
    for _, r in df_entities.iterrows():
        file_eid_map[str(r["数据来源文件"]).strip()][str(r["实体标识"]).strip()] = (
            str(r["实体所属类别"]).strip(), str(r["标准化名称"]).strip())

    syndrome_to_symptoms, syndrome_to_pathos = defaultdict(set), defaultdict(set)
    n_discard = 0
    for _, r in df_relations.iterrows():
        rel = str(r["关联关系"]).strip()
        if rel not in (REL_SYND_TO_SYMP, REL_SYND_TO_PATH):
            continue
        f = str(r["数据来源文件"]).strip()
        e1, e2 = str(r["实体1"]).strip(), str(r["实体2"]).strip()
        emap = file_eid_map.get(f, {})
        if e1 not in emap or e2 not in emap:
            n_discard += 1
            continue
        (t1, n1), (t2, n2) = emap[e1], emap[e2]
        if n1 == "未识别" or n2 == "未识别":
            continue
        if rel == REL_SYND_TO_SYMP and t1 == "证候" and t2 == "症状":
            syndrome_to_symptoms[n1].add(n2)
        elif rel == REL_SYND_TO_PATH and t1 == "证候" and t2 == "病机":
            syndrome_to_pathos[n1].add(n2)

    print(f"证候→症状 {len(syndrome_to_symptoms)} 个证候，"
          f"证候→病机 {len(syndrome_to_pathos)} 个证候，跨文件丢弃 {n_discard} 条")
    return syndrome_to_symptoms, syndrome_to_pathos


s2s, s2p = build_syndrome_maps(df_entities, df_relations)

n_sym = len({s for v in s2s.values() for s in v})
n_pat = len({p for v in s2p.values() for p in v})
n_syn = len(set(s2s) | set(s2p))
n_edge = sum(len(v) for v in s2s.values()) + sum(len(v) for v in s2p.values())
print("网络：症状 %d、证候 %d、病机 %d，合计 %d 个节点；%d 条边"
      % (n_sym, n_syn, n_pat, n_sym + n_syn + n_pat, n_edge))

G = nx.Graph()
for z, syms in s2s.items():
    for s in syms:
        G.add_edge(("症状", s), ("证候", z))
for z, pats in s2p.items():
    for p in pats:
        G.add_edge(("证候", z), ("病机", p))

pd.DataFrame([(a[0], a[1], b[0], b[1]) for a, b in G.edges()],
             columns=["源类型", "源名称", "目标类型", "目标名称"]
             ).to_excel(RES / "网络边表.xlsx", index=False)
pd.DataFrame([(n[0], n[1]) for n in G.nodes()], columns=["类型", "名称"]
             ).to_excel(RES / "网络节点表.xlsx", index=False)
print("已写出 网络边表 / 网络节点表")

## 1.5　证候桥接生成候选关联对（II-C）

只要症状 s 和病机 p 共享至少一个证候，该组合就进入候选集，共享证候数记作
n(s,p) = |C(s) ∩ C(p)|。顺带统计病机一侧的覆盖情况。

In [ ]:
def bridge_candidates(syndrome_to_symptoms, syndrome_to_pathos):
    """证候桥接，生成症状—病机候选对及其共享证候数。"""
    sp_counts = defaultdict(Counter)
    for synd, pathos in syndrome_to_pathos.items():
        symptoms = syndrome_to_symptoms.get(synd, set())
        if not symptoms or not pathos:
            continue
        for s in symptoms:
            for p in pathos:
                sp_counts[s][p] += 1
    rows = [(s, p, c) for s, cnt in sp_counts.items() for p, c in cnt.items()]
    cand = pd.DataFrame(rows, columns=["症状", "病机", "shared_syndrome_count"])
    print(f"桥接生成候选对 {len(cand)} 对")
    return cand


cand = bridge_candidates(s2s, s2p)
print("理论组合 %d，候选对占 %.2f%%" % (n_sym * n_pat, 100 * len(cand) / (n_sym * n_pat)))

std_pat = pd.read_excel(STD_DIR / "病机.xlsx", dtype=str).fillna("")
vocab = sorted(set(std_pat["标准化名"].astype(str).str.strip()))
patho_to_synd = defaultdict(set)
for z, pats in s2p.items():
    for p in pats:
        patho_to_synd[p].add(z)
n_one = sum(1 for p in patho_to_synd if len(patho_to_synd[p]) == 1)
print("标准病机词表 %d 个，进入网络 %d 个（%.1f%%），其中仅连 1 个证候 %d 个（%.1f%%）"
      % (len(vocab), n_pat, 100 * n_pat / len(vocab), n_one, 100 * n_one / n_pat))

## 1.6　前六项结构特征（II-D、表 I）

在保留节点类型的三层无向网络上计算。NPMI 的共现单元是证候，N 取证候节点总数；
AA 的度数取下界 2，RA 的取下界 1；PPR 按式 (2) 计算，以单个症状节点为唯一重启节点，
α = 0.85，取病机节点上的稳态概率。六项在全库候选上做 Min–Max 归一化。

In [ ]:
def compute_features(df_pairs, syndrome_to_symptoms, syndrome_to_pathos, graph):
    """算六项结构特征并做全库 Min–Max 归一化。"""
    symptom_to_synd, patho_to_synd = defaultdict(set), defaultdict(set)
    synd_degree = {}
    for z in set(syndrome_to_symptoms) | set(syndrome_to_pathos):
        syms, pats = syndrome_to_symptoms.get(z, set()), syndrome_to_pathos.get(z, set())
        synd_degree[z] = len(syms) + len(pats)
        for s in syms:
            symptom_to_synd[s].add(z)
        for p in pats:
            patho_to_synd[p].add(z)
    N = max(len(synd_degree), 1)

    npmi, jaccard, aa_list, ra_list = [], [], [], []
    for _, r in df_pairs.iterrows():
        S, P = symptom_to_synd[r["症状"]], patho_to_synd[r["病机"]]
        co = len(S & P)
        p_co, p_s, p_p = co / N, len(S) / N, len(P) / N
        pmi = np.log(p_co / (p_s * p_p)) if p_s * p_p > 0 else 0.0
        npmi.append(pmi / (-np.log(p_co)) if p_co > 0 else 0.0)
        jaccard.append(co / len(S | P) if len(S | P) > 0 else 0.0)
        aa_list.append(sum(1.0 / np.log(max(synd_degree[z], 2)) for z in (S & P)))
        ra_list.append(sum(1.0 / max(synd_degree[z], 1) for z in (S & P)))

    df = df_pairs.copy()
    df["NPMI"], df["Jaccard"] = npmi, jaccard
    df["AA"], df["RA"] = aa_list, ra_list

    ppr = {}
    for i, (s, plist) in enumerate(df.groupby("症状")["病机"].apply(list).items(), 1):
        pr = nx.pagerank(graph, alpha=PPR_ALPHA,
                         personalization={("症状", s): 1.0}, max_iter=PPR_MAX_ITER)
        for p in plist:
            ppr[(s, p)] = pr.get(("病机", p), 0.0)
        if i % 200 == 0:
            print("   PPR 进度 %d" % i)
    df["PPR"] = [ppr[(s, p)] for s, p in zip(df["症状"], df["病机"])]

    for c in ["shared_syndrome_count", "NPMI", "Jaccard", "AA", "RA", "PPR"]:
        v = df[c].astype(float)
        df[c + "_norm"] = (v - v.min()) / (v.max() - v.min() + 1e-12)
    return df


df_all_scores = compute_features(cand, s2s, s2p, G)
print(df_all_scores[["shared_syndrome_count", "NPMI", "Jaccard", "AA", "RA", "PPR"]]
      .describe().loc[["min", "max", "mean"]].round(6).to_string())
print("PPR 正值对数 %d / %d" % (int((df_all_scores["PPR"] > 0).sum()), len(df_all_scores)))
df_all_scores.to_excel(RES / "全库候选对_结构特征.xlsx", index=False)

## 1.7　专家抽样池与 250 对抽样（II-E）

以实体实例表为单位，按原始术语名统计出现次数，保留出现次数不低于 3 次的实体，
外加全部证候实体；在这个子集上重新做一次桥接得到候选池，用随机种子 42 抽 250 对。

In [ ]:
def sampling_pool(ent, rel):
    """频次筛选后重新桥接，得到专家抽样的候选池。"""
    ent, rel = ent.copy(), rel.copy()
    for c in ("实体标识", "数据来源文件", "实体具体名称", "实体所属类别"):
        ent[c] = ent[c].astype(str).str.strip()
    for c in ("实体1", "实体2", "数据来源文件"):
        rel[c] = rel[c].astype(str).str.strip()

    freq = ent["实体具体名称"].value_counts()
    keep = ent[(ent["实体具体名称"].map(freq) >= 3) | (ent["实体所属类别"] == "证候")]
    kept = {(f, i): n for f, i, n in zip(keep["数据来源文件"], keep["实体标识"],
                                         keep["实体具体名称"])}
    id2type = {(f, i): t for f, i, t in zip(ent["数据来源文件"], ent["实体标识"],
                                            ent["实体所属类别"])}

    sub = rel[[(f, i) in kept for f, i in zip(rel["数据来源文件"], rel["实体1"])]]
    sub = sub[[(f, i) in kept for f, i in zip(sub["数据来源文件"], sub["实体2"])]]

    synd2sym, synd2pat = defaultdict(set), defaultdict(set)
    for f, rr, e1, e2 in zip(sub["数据来源文件"], sub["关联关系"], sub["实体1"], sub["实体2"]):
        t1, t2 = id2type.get((f, e1)), id2type.get((f, e2))
        if rr == REL_SYND_TO_SYMP and t1 == "证候" and t2 == "症状":
            synd2sym[kept[(f, e1)]].add(kept[(f, e2)])
        elif rr == REL_SYND_TO_PATH and t1 == "证候" and t2 == "病机":
            synd2pat[kept[(f, e1)]].add(kept[(f, e2)])

    pool = set()
    for y in set(synd2sym) & set(synd2pat):
        for s in synd2sym[y]:
            for p in synd2pat[y]:
                pool.add((s, p))
    return pd.DataFrame(sorted(pool), columns=KEY_COLS), len(ent), len(keep), len(sub)


pool, n_ent, n_keep, n_valid = sampling_pool(df_entities, df_relations)
rng = np.random.RandomState(42)
sample = pool.iloc[sorted(rng.choice(len(pool), 250, replace=False))].reset_index(drop=True)
print("实体 %d → %d；保留合法关系 %d；抽样池 %d 对；按种子 42 抽 %d 对"
      % (n_ent, n_keep, n_valid, len(pool), len(sample)))
pool.to_excel(RES / "抽样池_1209对.xlsx", index=False)
sample.to_excel(RES / "抽样_250对.xlsx", index=False)

## 1.8　专家参考标注整理（II-E）

250 对经标准词表映射后得到 246 个唯一标准名对。同一标准名对由多行同义名归并撞出来的，
参考标签（专家 3 的定稿值）一致就保留一条，冲突则整组剔除，最后剩 243 对，
再与结构特征表按症状—病机键合并。三位专家在文件里记作专家1、专家2、专家3。

In [ ]:
def load_labels_aligned(path, std_map, df_scores):
    """读专家评分记录，做术语映射与冲突处理，再与结构特征表合并。"""
    df = pd.read_excel(path, dtype=str).fillna("")
    for c in KEY_COLS + ["专家1", "专家2", "专家3"]:
        df[c] = df[c].astype(str).str.strip()
    print("评分记录读取成功：%d 条（原始名称口径）" % len(df))

    df["症状_std"] = df["症状"].map(std_map["症状"]).fillna("")
    df["病机_std"] = df["病机"].map(std_map["病机"]).fillna("")
    n_fail = int(((df["症状_std"] == "") | (df["病机_std"] == "")).sum())
    if n_fail:
        print("%d 条术语不在映射表里，先用原始名兜底" % n_fail)
        df.loc[df["症状_std"] == "", "症状_std"] = df["症状"]
        df.loc[df["病机_std"] == "", "病机_std"] = df["病机"]
    print("映射后唯一标准名对：%d 个" % df.groupby(["症状_std", "病机_std"]).ngroups)

    mask_bad = ~df["专家3"].isin(LABEL_MAP.keys())
    if mask_bad.any():
        for idx in df[mask_bad].index:
            a_, b_ = df.loc[idx, "专家1"], df.loc[idx, "专家2"]
            if a_ == b_ and a_ in LABEL_MAP:
                df.loc[idx, "专家3"] = a_
        df = df[df["专家3"].isin(LABEL_MAP.keys())].reset_index(drop=True)
    df["y"] = df["专家3"].map(LABEL_MAP).astype(int)

    dup_mask = df.duplicated(subset=["症状_std", "病机_std"], keep=False)
    if dup_mask.any():
        keep_idx, adjudicate_idx = [], []
        for _, g in df[dup_mask].groupby(["症状_std", "病机_std"]):
            if g["专家3"].nunique() == 1:       # 参考标签一致，留一条
                keep_idx.append(g.index[0])
            else:                               # 标签冲突，导出待裁决并剔除
                adjudicate_idx.extend(g.index.tolist())
        if adjudicate_idx:
            df.loc[adjudicate_idx, KEY_COLS + ["症状_std", "病机_std", "专家1", "专家2", "专家3"]] \
              .to_excel(RES / "标注冲突_待人工裁决.xlsx", index=False)
            print("标签冲突 %d 条已导出待裁决（本次剔除）" % len(adjudicate_idx))
        df = pd.concat([df[~dup_mask], df.loc[keep_idx]]).reset_index(drop=True)

    df["症状"], df["病机"] = df["症状_std"], df["病机_std"]
    df = df.drop(columns=["症状_std", "病机_std"])
    print("对齐后标签分布:", df["y"].value_counts().sort_index().to_dict())

    df_m = df.merge(df_scores[KEY_COLS + FEATURE_COLS], on=KEY_COLS, how="left", indicator=True)
    n_hit = int((df_m["_merge"] == "both").sum())
    print("与结构特征表的匹配率：%d/%d (%.1f%%)" % (n_hit, len(df_m), 100 * n_hit / len(df_m)))
    return df_m[df_m["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)


df_labeled = load_labels_aligned(GOLD_DIR / "专家评分记录_250对.xlsx", std_map, df_all_scores)
df_labeled.to_excel(RES / "专家参考标注_243对.xlsx", index=False)
print("最终样本 %d 对" % len(df_labeled))

a = df_labeled["专家1"].map(LABEL_MAP).values.astype(int)
b = df_labeled["专家2"].map(LABEL_MAP).values.astype(int)
c = df_labeled["专家3"].map(LABEL_MAP).values.astype(int)
df_agree = pd.DataFrame([
    {"比较": tag,
     "精确一致率": round(float((x == z).mean()), 4),
     "±1 级一致率": round(float((np.abs(x - z) <= 1).mean()), 4),
     "线性加权 κ": round(float(cohen_kappa_score(x, z, weights="linear")), 4),
     "二次加权 κ": round(float(cohen_kappa_score(x, z, weights="quadratic")), 4)}
    for tag, x, z in [("专家1 vs 专家2", a, b), ("专家1 vs 专家3", a, c), ("专家2 vs 专家3", b, c)]])
df_agree.to_excel(RES / "专家间一致性.xlsx", index=False)
print(df_agree.to_string(index=False))

## 1.9　导出表 II 与候选关系概况（III-A）

In [ ]:
pd.DataFrame([("原始关系数", n_raw),
              ("去除原始名称重复后关系数", n_dedup),
              ("标准化整合后关系数", n_edge),
              ("症状节点数", n_sym),
              ("证候节点数", n_syn),
              ("网络病机节点数", n_pat),
              ("标准病机词表", len(vocab)),
              ("仅连接 1 个证候的病机", n_one),
              ("理论组合数", n_sym * n_pat),
              ("候选症状—病机对", len(cand)),
              ("候选占比", "%.2f%%" % (100 * len(cand) / (n_sym * n_pat))),
              ("抽样池", len(pool)),
              ("抽样对数（seed=42）", len(sample)),
              ("映射后唯一标准名对", 246),
              ("最终专家参考样本", len(df_labeled))],
             columns=["指标", "数值"]).to_excel(RES / "表II_数据整理与候选关系概况.xlsx", index=False)

print("已写出 网络边表 / 网络节点表 / 全库候选对_结构特征 / 抽样池_1209对 / 抽样_250对 / "
      "专家参考标注_243对 / 专家间一致性 / 表II_数据整理与候选关系概况")